# Lesson 3.1: Random Variables and Distributions

Build the probability distribution (PMF) of the sum of two dice as a table, find its mean and variance from the PMF, and check them by simulation.

Part of *Statistics from Scratch*. Run each cell with **Shift + Enter**, top to bottom.

If a package is missing on your computer, run the next cell once. In Google Colab everything is already installed.

In [ ]:
%pip install -q numpy scipy pandas statsmodels matplotlib

## 1. The PMF as a table

We count how many of the 36 equally likely pairs give each sum, then divide by 36. A pandas table keeps the values, the counts and the probabilities side by side.

In [1]:
import numpy as np
import pandas as pd
from itertools import product

sums = [a + b for a, b in product(range(1, 7), repeat=2)]
pmf = pd.Series(sums).value_counts().sort_index() / 36

table = pd.DataFrame({'ways': (pmf * 36).astype(int), 'P(X = x)': pmf.round(4)})
table.index.name = 'sum x'
print(table.to_string())

       ways  P(X = x)
sum x                
2         1    0.0278
3         2    0.0556
4         3    0.0833
5         4    0.1111
6         5    0.1389
7         6    0.1667
8         5    0.1389
9         4    0.1111
10        3    0.0833
11        2    0.0556
12        1    0.0278


## 2. Check it is a valid PMF and add the CDF

A valid PMF has probabilities between 0 and 1 that add to exactly 1. The running total (cumsum) is the CDF, which answers 'at most' questions at once.

In [2]:
print('total probability:', round(pmf.sum(), 10))

cdf = pmf.cumsum()
print(f'P(X <= 4)  = {cdf[4]:.4f}')
print(f'P(X >= 10) = {1 - cdf[9]:.4f}')

total probability: 1.0
P(X <= 4)  = 0.1667
P(X >= 10) = 0.1667


## 3. Mean and variance from the PMF

The mean is the sum of x times P(x). The variance is the sum of (x minus the mean) squared times P(x). These describe the whole population of possible rolls, so there is no n - 1 here.

In [3]:
x = pmf.index.to_numpy()
p = pmf.to_numpy()

mean = (x * p).sum()
var = ((x - mean) ** 2 * p).sum()

print(f'mean     = {mean:.4f}')
print(f'variance = {var:.4f}')
print(f'sd       = {var ** 0.5:.4f}')

mean     = 7.0000
variance = 5.8333
sd       = 2.4152


## 4. Check by simulation

Roll two dice 100,000 times with a seeded generator. The simulated mean and variance should sit close to the exact ones. ddof=0 matches the population formula above.

In [4]:
rng = np.random.default_rng(42)
rolls = rng.integers(1, 7, size=100_000) + rng.integers(1, 7, size=100_000)

print(f'simulated mean     = {rolls.mean():.4f}')
print(f'simulated variance = {rolls.var(ddof=0):.4f}')
print(f'simulated P(X = 7) = {(rolls == 7).mean():.4f}  (exact {pmf[7]:.4f})')

simulated mean     = 7.0055
simulated variance = 5.7999
simulated P(X = 7) = 0.1688  (exact 0.1667)


## Try this

- Change the dice to 8-sided ones (range(1, 9) and 64 pairs). Predict the new mean before you run it.
- Use one die instead of two. Predict the mean (3.5) and whether the variance is bigger or smaller than 5.83.

## The same idea in R, SPSS and Stata

**R**

```r
s <- as.vector(outer(1:6, 1:6, "+"))
pmf <- table(s) / 36
x <- as.numeric(names(pmf))
round(pmf, 4)
mu <- sum(x * pmf)
mu
sum((x - mu)^2 * pmf)
sqrt(sum((x - mu)^2 * pmf))

set.seed(42)
rolls <- sample(6, 1e5, TRUE) + sample(6, 1e5, TRUE)
c(mean(rolls), mean((rolls - mean(rolls))^2))
```

**SPSS** (Analyze > Descriptive Statistics > Frequencies (Statistics: Mean, Variance))

```
INPUT PROGRAM.
LOOP #a = 1 TO 6.
LOOP #b = 1 TO 6.
COMPUTE total = #a + #b.
END CASE.
END LOOP.
END LOOP.
END FILE.
END INPUT PROGRAM.
FREQUENCIES VARIABLES=total
  /STATISTICS=MEAN VARIANCE STDDEV.
```

**Stata**

```
clear
set obs 36
gen total = ceil(_n/6) + mod(_n - 1, 6) + 1
tabulate total
summarize total
display r(Var) * 35/36

clear
set seed 42
set obs 100000
gen rolls = ceil(6*runiform()) + ceil(6*runiform())
summarize rolls
```